# 🧭 Bússola Pública — Exploração da API da Câmara

**Objetivo deste notebook:** entender o terreno antes de codar o pipeline.

Vamos:
1. Fazer uma chamada simples sem nada da nossa lib (puro `requests`) — pra ninguém ficar achando que é mágica
2. Inspecionar a estrutura de resposta (campos `dados` e `links`)
3. Entender paginação HATEOAS (`links.rel='next'`)
4. Subir pro nosso `CamaraAPIClient`, que faz isso tudo automaticamente
5. Salvar os JSONs em `data/raw/` pra inspecionar offline

**Antes de rodar:**
```bash
python -m venv .venv && source .venv/bin/activate  # Linux/Mac
# .venv\Scripts\activate                          # Windows
pip install -r requirements.txt
cp .env.example .env
```

## 1. Chamada "crua" — só `requests`

Antes de usar abstrações, sentir o dado na mão. Endpoint `/deputados`, 2 itens só.

In [5]:
import requests
from pprint import pprint

BASE = "https://dadosabertos.camara.leg.br/api/v2"
r = requests.get(f"{BASE}/deputados", params={"itens": 2}, timeout=30)
print(f"HTTP {r.status_code}  •  {len(r.content):,} bytes")
payload = r.json()
print("\nChaves do envelope:", list(payload.keys()))
print(f"Quantos deputados nesta página: {len(payload['dados'])}")

ModuleNotFoundError: No module named 'requests'

### Inspecionar UM deputado

Os campos que aparecem aqui viram colunas em `dim_deputados`. Veja quais são úteis e quais a gente vai ignorar.

In [ ]:
pprint(payload['dados'][0], width=100, sort_dicts=False)

### Inspecionar os `links` — é assim que a paginação funciona

A API segue o padrão HATEOAS: cada página vem com links relativos (`self`, `first`, `next`, `last`).

Nossa estratégia de paginação: enquanto existir `rel='next'`, seguir buscando. Quando sumir, acabou.

In [ ]:
for link in payload['links']:
    print(f"{link['rel']:>6}  →  {link['href']}")

## 2. Usando o nosso `CamaraAPIClient`

Mesma chamada, agora com retry exponencial, paginação automática e persistência em `data/raw/`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.config import setup_logging
from src.extract.camara_api import CamaraAPIClient

log = setup_logging()
client = CamaraAPIClient()

In [ ]:
# Coleta TODOS os deputados em exercício (limite de 1 página = até 100 itens, só pra explorar)
out_path = client.save_raw("/deputados", max_pages=1)
print(f"\nArquivo salvo em: {out_path}")

## 3. Filtros úteis nos endpoints

A documentação Swagger lista os parâmetros aceitos por cada endpoint. Os mais importantes:

| Endpoint | Filtros úteis |
|---|---|
| `/deputados` | `siglaUf`, `siglaPartido`, `idLegislatura` |
| `/proposicoes` | `dataInicio`, `dataFim`, `siglaTipo`, `numero`, `ano` |
| `/votacoes` | `dataInicio`, `dataFim` |
| `/partidos` | (poucos filtros — universo pequeno) |

In [ ]:
# Exemplo: só deputados de SP
out = client.save_raw("/deputados", params={"siglaUf": "SP"}, max_pages=1)
print(out)

In [ ]:
# Exemplo: proposições dos últimos 7 dias
from datetime import date, timedelta
fim = date.today().isoformat()
inicio = (date.today() - timedelta(days=7)).isoformat()
out = client.save_raw(
    "/proposicoes",
    params={"dataInicio": inicio, "dataFim": fim},
    max_pages=2,
)
print(out)

## 4. Hipóteses de negócio que esses dados respondem

Antes de modelar tabelas, listar perguntas. Algumas que valem ouro:

- Quantas proposições por tema entram por semana?
- Quais 10 deputados são autores de mais proposições neste mês?
- Qual partido vota mais coeso?
- Quais categorias de despesa concentram mais gasto da cota parlamentar?
- Quanto tempo, em média, uma proposição leva entre apresentação e arquivamento?

Escrevam mais aqui em equipe — cada pergunta dessas vira um SELECT no Supabase na Sprint 5.

## 5. Próximos passos

- [ ] Rodar `python scripts/explore_api.py` no terminal e abrir os JSONs salvos
- [ ] Decidir, em equipe, quais campos da resposta vão pra cada tabela
- [ ] Definir as PKs (`id` da API é confiável) e relacionamentos
- [ ] Esboçar `sql/schema.sql` (Sprint 2)
